
# Implementando DBSCAN vs K-Means con `make_moons`

**Fecha:** 2025-08-26 19:14

---

## 🎯 Requisitos de la práctica
1. Generar datos de prueba con `make_moons()`.
2. Aplicar **DBSCAN** con diferentes valores de `eps`.
3. Comparar los **clusters** resultantes con el algoritmo **K-Means**.

---

## 🧠 Idea general
- `make_moons` produce dos medias lunas entrelazadas (no linealmente separables).
- **K-Means** tiende a particionar por **fronteras lineales** (esféricas por distancia euclídea), por lo que **no** se adapta bien a formas no convexas.
- **DBSCAN** identifica **densidades** (clusters de alta densidad separados por regiones de baja densidad) y **ruido** (`label = -1`), lo que suele funcionar mejor en datos con formas curvas/irregulares.

**Hiperparámetros clave en DBSCAN**:
- `eps`: radio de vecindad (distancia máxima para considerar puntos vecinos).
- `min_samples`: cantidad mínima de vecinos para considerar un punto **núcleo**.

> Objetivo: ver cómo cambia el resultado de DBSCAN al variar `eps` y compararlo con K-Means (k=2).



## ⚙️ Requisitos
```bash
pip install -U numpy matplotlib scikit-learn
```


In [ ]:

# Imports y versiones
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
import pandas as pd
import sys, sklearn

print(f"Python: {sys.version.split()[0]} | numpy: {np.__version__} | matplotlib: {plt.matplotlib.__version__} | sklearn: {sklearn.__version__}")



## 1) Generar datos con `make_moons()`

- Agregamos **ruido** para dificultar la tarea.
- Estandarizamos para que los métodos basados en distancia funcionen mejor.


In [ ]:

# Generar datos
X, y_true = make_moons(n_samples=600, noise=0.08, random_state=42)

# Estandarizar
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

# Vista rápida
print("Shape:", X_std.shape)
print("Primeras 5 filas:\n", np.round(X_std[:5], 3))



## 2) Funciones auxiliares

- `plot_clusters`: visualiza etiquetas de cluster (cada figura es independiente).  
- `evaluate_clustering`: calcula **ARI**, **NMI**, y **silhouette** (si aplica).  
- `run_dbscan_sweep`: ejecuta DBSCAN para una lista de `eps`.


In [ ]:

def plot_clusters(X, labels, title="Clusters"):
    """Crea una figura de dispersión 2D. Cada llamada genera una figura independiente."""
    plt.figure(figsize=(6, 5))
    plt.scatter(X[:, 0], X[:, 1], c=labels, s=25, alpha=0.9, edgecolor="k")
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.tight_layout()
    plt.show()


def evaluate_clustering(X, y_true, labels):
    """Devuelve métricas ARI, NMI, Silhouette (si hay >=2 clusters distintos y < n muestras en cada)."""
    unique = np.unique(labels)
    # ARI / NMI no requieren condiciones adicionales
    ari = adjusted_rand_score(y_true, labels)
    nmi = normalized_mutual_info_score(y_true, labels)
    # Silhouette:
    sil = np.nan
    # silhouette_score requiere al menos 2 clusters y no todos los puntos en el mismo cluster
    if len(unique) >= 2 and len(unique) < len(labels):
        try:
            sil = silhouette_score(X, labels)
        except Exception:
            sil = np.nan
    return ari, nmi, sil


def run_dbscan_sweep(X, y_true, eps_list, min_samples=5):
    rows = []
    for eps in eps_list:
        db = DBSCAN(eps=eps, min_samples=min_samples)
        labels = db.fit_predict(X)
        ari, nmi, sil = evaluate_clustering(X, y_true, labels)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = int(np.sum(labels == -1))
        rows.append({
            "model": f"DBSCAN(eps={eps}, min_samples={min_samples})",
            "eps": eps,
            "min_samples": min_samples,
            "clusters": n_clusters,
            "noise_pts": n_noise,
            "ARI": ari,
            "NMI": nmi,
            "Silhouette": sil
        })
        # Plot por configuración
        plot_clusters(X, labels, title=f"DBSCAN | eps={eps}, min_samples={min_samples} | clusters={n_clusters}, noise={n_noise}")
    return pd.DataFrame(rows)



## 3) Aplicar DBSCAN con diferentes valores de `eps`

Probamos varios `eps` para observar cómo cambia:
- Número de clusters detectados
- Cantidad de ruido
- Métricas ARI, NMI y Silhouette


In [ ]:

eps_list = [0.1, 0.15, 0.2, 0.25, 0.3]
dbscan_results = run_dbscan_sweep(X_std, y_true, eps_list=eps_list, min_samples=5)
dbscan_results



## 4) Comparación con K-Means (k=2)

Entrenamos **K-Means** con `k=2` (dos lunas) y comparamos métricas y visualización.


In [ ]:

kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
k_labels = kmeans.fit_predict(X_std)
k_ari, k_nmi, k_sil = evaluate_clustering(X_std, y_true, k_labels)
plot_clusters(X_std, k_labels, title=f"K-Means | k=2 | ARI={k_ari:.3f}, NMI={k_nmi:.3f}, Sil={k_sil:.3f}")

kmeans_row = pd.DataFrame([{
    "model": "KMeans(k=2)",
    "eps": np.nan,
    "min_samples": np.nan,
    "clusters": len(np.unique(k_labels)),
    "noise_pts": 0,
    "ARI": k_ari,
    "NMI": k_nmi,
    "Silhouette": k_sil
}])
kmeans_row



## 5) Tabla comparativa (DBSCAN vs K-Means)

Observa especialmente:
- **DBSCAN** debe capturar mejor la forma de **dos lunas**, a menudo con **mayor ARI/NMI** que K-Means.  
- **K-Means** tiende a cortar por fronteras lineales, lo que puede mezclar puntos de ambas lunas.


In [ ]:

results = pd.concat([dbscan_results, kmeans_row], ignore_index=True)
results.sort_values(by=["model", "eps"], inplace=True, na_position="last")
results.reset_index(drop=True, inplace=True)
results



## ✅ Guía de interpretación
- Si `eps` es **muy pequeño**, DBSCAN considera pocos vecinos → muchos **ruidos** y **pocos clusters**.
- Si `eps` es **muy grande**, todos los puntos se conectan → **un único cluster** (o pocos, silueta baja).
- Busca un rango de `eps` que **maximice ARI/NMI** y mantenga **ruido moderado**.
- **K-Means** sirve como **línea base**, pero no modela bien formas **no convexas**.

## 🌱 Extensiones sugeridas
- Barrer `min_samples` (p. ej. 3–10) y hacer un **heatmap** de ARI/NMI por (`eps`, `min_samples`).
- Probar con `make_circles` o datasets reales de formas no lineales.
- Experimentar con otras **métricas de distancia** o **escalados** (RobustScaler, MinMaxScaler).
